# 04 – Modelling: task (c), penalty minutes per player

Fourth phase of **CRISP-DM (Modelling)** for requirement **(c)**: predict how many regular-season penalty minutes (PIM) each skater will accumulate in season $t$, the day before the season starts (goal **G4**, Business Understanding section 3).

| Phase | Notebook |
|---|---|
| 1. Business Understanding | `00_business_understanding.ipynb` |
| 2. Data Understanding | `01_eda.ipynb`, `02_statistical_analysis.ipynb` |
| 3. Data Preparation | `03_data_preparation.ipynb` |
| 4. Modelling, task (c) | `04_modelling_pim.ipynb` (this notebook) |

**What this notebook takes from the previous phases:**

| Decision | Where it was made | Used here |
|---|---|---|
| Unit: one row per NHL skater-season, 1982–2009, 20,539 rows | DP 3, 4 | `dataset_pim_pruned.csv` |
| 13 features (11 + flags `is_rookie`, `is_returning`), 5 `log_` versions | DP 7, 8, 9 | read from `pim_feature_metadata.json` |
| Evaluation: walk-forward, test seasons 2000–2009 without 2004 | BU 3 | every result of this notebook |
| Preprocessing fitted on the training seasons only: median imputation → winsorizing (1–99%) → log(1 + x) → standardizing; trees get only the imputation | DP 5.3, 8.1.2 | the `prepare` function (section 1) |
| Development sample: 25% of the training seasons for the linear model; Gradient Boosting still improves at 100% | DP 10.2 | tuning (section 5) |
| Feature selection: all 13 features beat filter / wrapper for the Poisson model | DP 12 | starting feature set (checked again for trees in section 10) |
| Success criterion: MAE < 19.5 minutes and Spearman > 0.71 (the *same as last season* baseline) | BU 2, 3 | section 7 |

**How this notebook is organised:** each section answers one criterion of the predictive part of the grading rubric.

| Section | Rubric criterion | Level aimed at, and how |
|---|---|---|
| 2 Baselines | baseline | suitable baseline: *same as last season*, what the users already do (plus a random baseline for the classification view) |
| 3 Performance measures | performance measure; interpretation | aligned with the DM goals (MAE, Spearman) **and** the data (Poisson deviance for a skewed count, MAE of the top 10%); advanced measures (D², ROC-AUC, PR-AUC) |
| 4 Algorithms | diversity of algorithms | 6 algorithms with different language biases |
| 5 Parameter tuning | parameter tuning | systematic search with a time-respecting validation |
| 6 Algorithm behaviour | understanding algorithm behaviour | all algorithms, with the effect of their parameters (validation curves) |
| 7 Results on the test seasons | training vs test; other factors (time) | correctly separated, 9 splits; time order, prediction time and drift taken into account |
| 8 Statistical significance | analysis of results | Friedman + Nemenyi + Wilcoxon |
| 9 Overfitting | analysis of overfitting | train vs test gap per model and per parameter, plus the optimism of the tuning itself |
| 10 Model improvement | model improvement | iterations driven by explicit performance goals and error analysis |
| 11 Classification view (high-PIM players) | diversity of tasks | regression **and** classification |
| 12 Feature importance | feature importance | permutation importance on test, interpreted in hockey terms |
| 13 White-box models | analysis of white-box models | GLM coefficients and tree rules, interpreted in hockey terms |
| 14 Final model and conclusions | | |

## 1. Setup

**What to do:**
1. Imports and global settings (seed, warnings, figure folder `../docs/figures/mod/`).
2. Read `dataset_pim_pruned.csv` and `pim_feature_metadata.json`. Take the lists of features, `log_` columns and evaluation seasons **from the metadata**, never written again by hand.
3. Read `dataset_pim_full.csv` as well, only for the columns needed by the baselines (`pim_prev`, `played_prev`), which section 7 of DP removed from the pruned file.
4. Bring the `prepare(train, test, version, log_columns)` function of DP section 8 (copy it, or better, move it to `src/` and import it in both notebooks, so there is only one version).
5. Write **one evaluation function** used by every experiment of the notebook. For each test season $t$ in `eval_years`: train on `year < t`, predict `year == t`, and keep:
   - the predictions of the test season (with `year`, `playerID`, `target_pim`, `is_rookie`), to compute any metric afterwards;
   - the predictions on the **training** seasons too, needed for the overfitting analysis (section 9);
   - the training time.

   It returns a table with one row per test player-season. Every metric is then computed per season (9 values) and on all seasons together.

**Why one function:** every algorithm, baseline and parameter value is then evaluated on exactly the same splits, with the same preprocessing. That makes the comparisons of sections 7–8 fair and **paired** (same 9 seasons, same players).

## 2. Baselines

A model is only useful if it beats what the user could do **without** data mining. We define the baselines first, before any model, and evaluate them with the same function and the same seasons.

| Baseline | Prediction for player $i$ in season $t$ | Why |
|---|---|---|
| B0 – global mean | mean PIM of the training seasons | the weakest reference: a model that learns nothing (DP 8.1.2 already showed that an unscaled Poisson model falls to this level) |
| B1 – same as last season | PIM of season $t-1$; for rookies (no $t-1$), the median PIM of the rookies of the training seasons | the business baseline of BU section 2 (MAE 19.5, Spearman 0.71) |
| B2 – career rate × last games | `career_pim_pg_before` × `gp_prev` (rookies as in B1) | a "smart" domain baseline: rate × games, the decomposition of DP 4.1.7, without learning anything |

**What to do:** compute the three baselines and their metrics (section 3), per season and in total. Check that B1 gives the values of BU section 2. If not, find the reason (e.g. different players included) before going on.

**What to discuss:** B1 is the **suitable** baseline because it is what the users already do. B0 only shows that the model learns something. B2 shows whether ML adds anything beyond a simple formula a domain expert could write.

## 3. Performance measures

The target is a **count** (minutes, ≥ 0) with a long right tail (skewness 2.03, DP 4.1.7): many players with few PIM and a few enforcers with 150+. The measures must match both the **DM goal** (BU 3) and these **data characteristics**:

| Measure | Role | Why |
|---|---|---|
| **Mean Poisson deviance** | main measure for **choosing and tuning** models | the proper error for a count: it judges an error relative to the expected value (a miss of 20 minutes counts more for a player expected at 10 than for one expected at 150), so a few enforcers do not dominate it the way they dominate the squared error |
| **D² (deviance explained)** = 1 − deviance(model) / deviance(B0) | readable version of the deviance | the "R²" of count models: 0 = no better than the mean, 1 = perfect |
| **MAE** (minutes) | business measure | users read it directly: "off by x minutes per player" (BU 1.11); criterion < 19.5 |
| **Spearman** between predicted and real PIM | business measure | users sort the list to find the most penalized players; criterion > 0.71 |
| **MAE on the top 10% of PIM** of each season | data characteristic | BU 1.11: errors on the high-PIM players matter most; a model can have a good overall MAE and still predict 60 for a player with 200 |
| **MAE for rookies vs veterans** | data characteristic | rookies have no history (missing features), so errors are expected to be larger |

RMSE is reported but not used to choose: with this tail it is driven by a handful of enforcers. Classification measures for the high-PIM view (ROC-AUC, PR-AUC, precision@k) are in section 11.

**What to do:** one function `scores(pred_table)` that returns all the measures above, per season and overall. Use it for the baselines now and for every model later.

**What to discuss (interpretation):** what a D² of e.g. 0.45 means, why MAE and deviance can disagree (one is in minutes, the other relative), and why Spearman can be high while the MAE of the top 10% is bad (the order is right but the high values are shrunk towards the mean, which is typical of regression).

## 4. Algorithms (iteration 1: default parameters)

We use algorithms with **clearly different language biases** (the form of the function each one can represent), so the comparison says something about the problem and not only about parameters:

| # | Algorithm | Language bias | Preprocessing (DP 8) | Loss for a count |
|---|---|---|---|---|
| 1 | **Poisson regression** (`PoissonRegressor`) | linear in log space: $E[\text{PIM}] = e^{b_0} \cdot \prod x_j^{b_j}$ (multiplicative) | winsor → log → standardize | Poisson |
| 2 | **Decision tree** (`DecisionTreeRegressor`, `criterion="poisson"`) | axis-parallel rectangles, constant inside each leaf | imputation only | Poisson |
| 3 | **k-nearest neighbours** (`KNeighborsRegressor`) | no model: local average of similar players (instance-based) | winsor → log → standardize | (mean of the neighbours) |
| 4 | **Random Forest** (`RandomForestRegressor`, `criterion="poisson"`) | average of many deep trees (bagging + random features) | imputation only | Poisson |
| 5 | **Gradient Boosting** (`HistGradientBoostingRegressor`, `loss="poisson"`) | sum of many small trees, each correcting the previous ones | imputation only | Poisson |
| 6 | **Neural network** (`MLPRegressor`) | smooth non-linear function (layers of neurons) | winsor → log → standardize | squared error on `log1p(PIM)`, back-transformed |

Possible variant of (1): **Negative Binomial GLM** (`statsmodels`), if the deviance shows **overdispersion** (variance much larger than the mean, likely with the enforcers). It is the same model with one extra parameter for the variance, so it also helps the white-box analysis (section 13).

**Notes:**
- The MLP in scikit-learn has no Poisson loss, so it learns `log1p(PIM)` and the prediction is `expm1(·)`. This predicts something close to the **median** rather than the mean, so it may underestimate on average. That is worth a comment when the results come in.
- kNN and Random Forest are the slowest on ~20,000 rows; if needed, use the development sample of DP 10.2 for tuning (section 5), never for the final test.

**What to do:** run all 6 with default parameters (only `random_state` fixed, `max_iter` large enough) through the evaluation function, and put them in one table with the baselines.

**What to discuss:** first impressions. Do the flexible models (trees, ensembles) beat the linear one? Does any model fail to beat B1? (Expected: the single tree with defaults overfits badly, see section 9.)

## 5. Parameter tuning

**Systematic approach:** grid search (small grids) or randomized search (large grids) over the parameters that control each algorithm's **complexity**, scored with the **Poisson deviance** on a validation that respects time.

**Validation for the tuning:** the test seasons 2000–2009 cannot be used to choose parameters, or the final test score becomes optimistic. Two correct options:

| Option | How | Cost |
|---|---|---|
| A – tune once, before the test seasons | inner walk-forward **inside 1982–1999**: train on the seasons before $s$, validate on $s$, for $s$ = 1997, 1998, 1999 (the `inner_splits` of DP 12); freeze the best parameters for all 9 test seasons | 1 search per algorithm |
| B – nested | repeat the search inside each of the 9 outer steps | 9 searches per algorithm |

**Recommended: option A.** It is leak-free (the tuning only sees seasons before 2000) and 9× cheaper. Option B can be done afterwards for the best model only, to check that the parameters are stable.

**Grids (starting point; adjust after seeing the validation curves of section 6):**

| Algorithm | Parameters |
|---|---|
| Poisson | `alpha` (L2 penalty): 1e-4 … 10 (log scale) |
| Decision tree | `max_depth` 2–20, `min_samples_leaf` 1–500 |
| kNN | `n_neighbors` 5–300, `weights` (uniform / distance) |
| Random Forest | `n_estimators` 200–500, `max_features` (0.3, 0.5, 1.0), `min_samples_leaf` 1–100 |
| Gradient Boosting | `learning_rate` 0.01–0.3, `max_iter` 100–1000 (with early stopping), `max_leaf_nodes` 7–63, `min_samples_leaf`, `l2_regularization` |
| MLP | `hidden_layer_sizes` ((16,), (32,), (64, 32)), `alpha` 1e-5–1e-1, `learning_rate_init` |

**Development sample:** DP 10.2 showed that the Poisson model is stable from 25% of the rows, while Gradient Boosting still improves at 100%. Use 25% (stratified by season) for the searches of the slow models if time demands it, and **say so**; the final models are always retrained on 100%.

**What to do:** one search per algorithm, save the best parameters, the **best inner-validation deviance** (needed in section 9 to measure how optimistic the tuning was) and the full search results (needed for the curves of section 6). Then re-run section 4 with the tuned parameters (iteration 2 of section 10).

## 6. Understanding algorithm behaviour

For each algorithm, a **validation curve**: training deviance and validation deviance (inner seasons 1997–1999) as a function of its main complexity parameter, the others fixed at the best value:

| Algorithm | Parameter on the x-axis | Expected behaviour |
|---|---|---|
| Poisson | `alpha` | small alpha = free coefficients; large alpha = all coefficients pulled to 0, predictions tend to the mean (underfitting). With 20,000 rows and 13 features, little effect is expected until alpha is large |
| Decision tree | `max_depth` | training error always falls; validation error falls then rises (overfitting): the classic U |
| kNN | `n_neighbors` | small k = noisy (each prediction copies a few players); large k = everything tends to the mean |
| Random Forest | `min_samples_leaf`, `n_estimators` | more trees never overfit, they only stabilise; leaf size controls smoothness |
| Gradient Boosting | `max_iter` × `learning_rate` | each new tree lowers the training error; the validation error stops improving and slowly rises; a smaller learning rate needs more trees |
| MLP | `alpha`, hidden size | larger network = more flexible; alpha controls overfitting |

**What to discuss, per algorithm:** does the curve match the expected behaviour? Where is the bias–variance balance? Why does an algorithm do well or badly **on this problem**? For example:
- The **multiplicative** form of the Poisson model fits PIM ≈ games × rate (DP 8.1.2), so a linear model can be very hard to beat here.
- A **single tree** gives the same prediction to everyone in a leaf, so it can only predict a few distinct values, which is bad for Spearman.
- **kNN** suffers when a few features carry most of the signal and the others add distance noise.
- The **ensembles** can find interactions (e.g. defenders vs forwards, rookies vs veterans) that the linear model only sees if we create them.

## 7. Results on the test seasons

Final comparison: baselines + 6 tuned algorithms, walk-forward on 2000–2009 without 2004 (9 test seasons, ~7,900 test player-seasons).

**Performance estimation (training vs test, other factors):**
- **Correctly separated data, multiple splits:** each test season is predicted by a model trained only on earlier seasons, and preprocessing, imputation and tuning are fitted without the test seasons. That gives **9 splits**, not one.
- **Time taken into account:** a random K-fold would train on 2008 to predict 2003 and put the same player's consecutive seasons on both sides, so the score would not be what users get in practice. Walk-forward reproduces the real use: predict next season from the past.
- **Prediction time:** every feature is known the day before season $t$ (DP 2.6.2). `eval_gp` and `eval_pim_pg` (games and rate of season $t$) are **never** features; they are only used to analyse the errors.

**What to do:**
1. Table: one row per model, the measures of section 3 as mean ± std over the 9 seasons, plus the overall value. Mark the best per column.
2. Figure: score per season, one line per model (shows whether one model wins in every season or only on average, and whether there is a trend over time).
3. Compare with the success criteria: MAE < 19.5 and Spearman > 0.71 (B1).

4. **Drift over time** (DP 2.6.3, "is old data still representative?"): plot the mean real PIM per season next to the mean predicted PIM. If the league's PIM level falls over time (less fighting), a model trained on 1982–1999 will **overpredict** the later seasons: a systematic error that comes from time, not from the model. Report the mean error (bias), not only the MAE.

**What to discuss:** which model wins, by how much compared with B1, whether the ranking of the models is the same for every measure, and which seasons are hard (e.g. 2005, the first season after the lockout, with new rules against obstruction: many more minor penalties).

## 8. Statistical significance

With only 9 test seasons, a difference of 0.1 in deviance may be just chance. We test it:

| Test | Question | Data |
|---|---|---|
| **Friedman** | do the models differ at all? | deviance of each model in each of the 9 seasons (rank the models within each season) |
| **Nemenyi** post-hoc + **critical difference diagram** | which pairs differ? | same ranks |
| **Wilcoxon signed-rank** (paired) | is the best model better than B1? than the second best? | the 9 per-season differences |

Optional, with more power: Wilcoxon on the ~7,900 **per-player** paired errors (best vs second). Be careful: players of the same season are not fully independent, so this p-value is optimistic. Say it.

**What to discuss:** with n = 9 the tests have low power, so "not significant" does not mean "equal". Separate practical significance (minutes of MAE that matter to a user) from statistical significance. If two models are statistically tied, prefer the simpler / more interpretable one (relevant for section 14).

## 9. Overfitting

**Estimate:** for each model, the deviance on the **training** seasons (predicting the data it was fitted on) vs the deviance on the **test** season, in each walk-forward step. The gap (test − train) is the overfitting.

**What to do:**
1. Table / bar chart: training vs test deviance per model (defaults and tuned side by side).
2. Use the validation curves of section 6: where the training curve keeps falling while the validation curve rises.
3. Learning curve for the best model (reuse DP 10.2): a gap that shrinks with more data = variance; two curves that meet high = bias.

**What to discuss (correct analysis):**
- A gap alone is not bad; it is bad when it **grows with complexity while the test error gets worse**. A Random Forest always has a large gap (deep trees memorise), yet can still be the best on test.
- The default decision tree should show the largest gap (it can grow until each leaf has one player), and tuning (depth, leaf size) should close most of it.
- The Poisson model should have almost no gap (13 coefficients, 20,000 rows): if it loses, it is bias, not variance.
- **Overfitting of the tuning itself:** compare the best inner-validation deviance of section 5 with the deviance of the same model on the test seasons. If the test is much worse, part of the "best" parameters was luck on 1997–1999 (choosing the best of many combinations is optimistic). The more combinations an algorithm had, the larger this optimism tends to be.
- Part of the test error is not overfitting but **change over time** (the 2005 rule change): the training error is measured on past seasons, the test error on a new one.

## 10. Model improvement

Development guided by **performance goals**, set before the experiments:

| Goal | Measure | Target |
|---|---|---|
| beat the business baseline | MAE / Spearman | < 19.5 min and > 0.71 (B1) |
| be clearly better than the reference model | Poisson deviance | below the tuned Poisson regression, significant in section 8 |
| do not miss the enforcers | MAE of the top 10% of PIM | lower than B1 and than the previous iteration |
| do not fail the rookies | MAE of the rookies | lower than the rookie median of B1 |

Each iteration starts from a hypothesis found in the previous results (usually in the error analysis), makes one change, and keeps it only if it improves the deviance on the **inner validation** (not on the test seasons). Log every iteration:

| Iteration | Hypothesis (from where) | Change | Validation deviance | Kept? |
|---|---|---|---|---|
| 1 | | 6 algorithms, defaults (section 4) | | |
| 2 | defaults over- or underfit (section 6) | tuning (section 5) | | |
| 3 | trees could use fewer / different features than the Poisson model (DP 12 tested only Poisson) | feature selection for the best tree model | | |
| 4 | *from the error analysis below* | | | |
| 5 | different models make different errors | average of Poisson + best ensemble | | |

**Error analysis (source of the next hypotheses):** residuals of the best model by group:
- rookies vs veterans (missing history);
- forwards vs defenders;
- by real PIM decile (are the enforcers shrunk towards the mean?);
- by real games played `eval_gp` (a player injured for half the season gets half the PIM; that is unpredictable noise, and it shows how much of the error no model can remove);
- by season (2005).

**Candidate changes for iteration 4** (choose from what the errors show):
- a **two-stage model**: predict games played and PIM per game separately, then multiply (this follows the domain decomposition of DP 4.1.7);
- new interactions for the linear model (e.g. `is_defense × rate`);
- the Negative Binomial variant if the enforcers' errors show overdispersion;
- a separate model, or different imputation, for rookies;
- **sliding window instead of expanding window** (train only on the last N seasons, e.g. N = 5, 10): if old seasons are less representative (drift, section 7), forgetting them can help. This is a time-related experiment, tested on the inner validation like the others.

**Stop rule:** stop when an iteration no longer improves the inner deviance by more than the season-to-season noise (the std of section 7), and say which goals were met and which were not.

## 11. Classification view: who will be a high-PIM player?

Users care most about the **most penalized players** (BU 1.9 and 1.11: coaches planning the penalty kill, fantasy players, over/under bets). That gives a second data mining task on the same data, a **binary classification**:

> target: 1 if the player is in the **top 10% of PIM of season t** (≈ the enforcers), 0 otherwise.

Defined per season, so it does not depend on the PIM level of each era. About 10% positives → imbalanced: use the decision of DP 11 (class weights) and **no accuracy**.

**Two ways, compared:**
1. **Ranking from the regression:** use the predicted PIM of the best model as a score.
2. **Dedicated classifiers:** logistic regression, decision tree, Gradient Boosting classifier (same features, same walk-forward, light tuning as in section 5).

**Baselines:** a random ranking (PR-AUC = 0.10, ROC-AUC = 0.5) and the **domain rule** "the players who were in the top 10% last season will be again" (rookies = 0). The rule is the suitable baseline: it is what a fantasy player or a coach would do.

**Measures (advanced):** ROC-AUC, PR-AUC (random = 0.10), **precision@k** (of the k players predicted highest in a season, how many are really in the top 10%; k = number of real positives), and the confusion matrix at the threshold chosen on the inner validation. The threshold depends on the user: a bettor wants high precision (few, sure picks), a coach planning the penalty kill wants high recall (miss no enforcer). Show the precision–recall curve and two operating points.

**Significance:** Wilcoxon on the 9 per-season PR-AUCs (best classifier vs the regression ranking, and vs the domain rule).

**What to discuss:** does a dedicated classifier find the enforcers better than sorting by the regression? What does ROC-AUC = 0.9 mean for a coach (the probability that a random enforcer is ranked above a random non-enforcer)? Why PR-AUC is more demanding here.

## 12. Feature importance

**Method:** **permutation importance** on the **test** seasons, for the best model and for the Poisson model: shuffle one feature and measure how much the deviance gets worse. It is model-agnostic, so the models can be compared, and it is measured on unseen data. The impurity importance of the trees is biased towards continuous features with many values, so it is not used for conclusions. Optionally, SHAP values for the best tree model.

Grouped importance: `pim_pg_prev_shrunk` and `career_pim_pg_before` are correlated (both measure how penalized the player usually is). Permuting one alone underestimates both, so also permute them **together** as a group "PIM history".

**What to do:** importance per test season (mean ± std over the 9), bar chart, and partial dependence plots (PDP) for the 3–4 most important features.

**What to discuss, relating to the domain:**
- PIM history dominates: a player's role (enforcer or not) is stable from season to season.
- `gp_prev` and `career_gp_before`: more games give more PIM, and veterans are trusted with regular minutes.
- `is_defense`, `weight`, `height`: physical players and defenders take more penalties.
- `career_pts_pg_before`: scorers usually take fewer PIM (a different role).
- `league_pim_pg_prev`: the era effect (fighting declining over time).
- `team_pim_pg_rel_prev`: the team's style or the coach's tolerance.

Check the PDPs against this: does the direction match the domain? If a feature acts against hockey sense, look for the reason (correlation with another feature, a data problem) before trusting it.

## 13. White-box models

**Poisson regression (coefficients):** with standardized features, $e^{b_j}$ is the factor by which the expected PIM is multiplied when feature $j$ increases by one standard deviation, the others fixed. For the `log_` features, $b_j$ is close to an **elasticity**: +1% in the rate gives about +$b_j$% PIM. With `statsmodels` (same features, same preprocessing, fitted on all seasons before the last test season) we also get confidence intervals and the dispersion check: Pearson χ² / df ≫ 1 → overdispersion → Negative Binomial.

**Interpreting correctly:** a coefficient is the effect **with the other features fixed**. Correlated features (e.g. `pim_pg_prev_shrunk` and `career_pim_pg_before`, `height` and `weight`) share their effect, so one can get a small or even "wrong" sign while the pair together is strong. Check this against the permutation importance of the group (section 12) before concluding that a feature does not matter. Also, the coefficients describe association, not cause: being heavy does not *make* a player take penalties, it marks the physical role.

**Shallow decision tree** (depth 3–4, Poisson criterion): draw it (`plot_tree`) and read the first splits as rules, e.g. "if career PIM/game > x and games last season > y → expected PIM z". Compare its test score with the best model: that difference is the **price of interpretability**.

**What to discuss, relating to the domain:**
- Are the signs of the coefficients what a coach would expect (section 12)? Are the effects of the same size in both white-box models?
- Is the coefficient of `log_gp_prev` close to 1 (PIM proportional to games, the decomposition of DP 4.1.7)?
- Which rules of the tree a GM could use directly ("players with these characteristics will be over 100 PIM").

## 14. Final model and conclusions

**What to do:**
1. Choose the final model with the results of sections 7–8 (if tied, the simpler one).
2. Retrain it on all seasons up to 2008 and show the prediction for 2009: the 20 players predicted with the most PIM vs their real PIM.
3. Back to the business criteria (BU 2 and 3): is MAE < 19.5 and Spearman > 0.71? By how much does the model beat *same as last season*?

**Summary table** *(to fill at the end)*:

| | B1 same as last season | Poisson | Best model | Criterion |
|---|---|---|---|---|
| Poisson deviance | | | | lower |
| D² | | | | higher |
| MAE (minutes) | | | | < 19.5 |
| Spearman | | | | > 0.71 |
| MAE top 10% | | | | lower |
| PR-AUC high-PIM | | | | > 0.10 |

**Limitations to state:** games played in season $t$ are unknown (injuries, trades), so part of the error cannot be removed. Rule changes after 2009 are not in the data. Rookies have no history.